# 🎬 Spot Republic · Editor con IA

Toma los videos DJI Mimo de tu carpeta **Server** de Drive y crea un spot cinematográfico:

1. **Selección con IA:** CLIP (ViT-H-14 en GPUs grandes) puntúa la estética, la nitidez, la exposición y la estabilidad de cada momento, y se queda con los mejores planos. Quita los clips duplicados.
2. **Montaje al ritmo:** los cortes caen en los beats fuertes de `spot_republic_soundtrack_master.mp4`, con push-in/pull-out y destellos en los golpes.
3. **Corrección de color:** teal & orange, warm film, cool night o b/n noir, con grano, viñeta y barras 2.39:1. El HDR/HLG de DJI se convierte a Rec.709.
4. **Textos en perspectiva 3D:** el título gira hacia la cámara y los **pensamientos del editor** flotan inclinados en el espacio. Los escribe un **modelo de IA local** (Qwen3.8-27B con visión, servido con vLLM en tu GPU) mirando cada plano, o Claude si lo eliges.
5. **Subtítulos con IA:** Whisper large-v3, palabra por palabra, estilo karaoke.
6. **Formato vertical 9:16:** YOLO sigue a las personas para reencuadrar (Reels/TikTok).

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo → **GPU H100** (o A100 80 GB) para el modelo local. La 1.ª vez descarga ~56 GB del modelo. (Opcional) para usar Claude: 🔑 Secretos → `ANTHROPIC_API_KEY` y `MOTOR_IA = claude`. Luego **Entorno de ejecución → Ejecutar todo**.
Los resultados quedan en `Server/Edicion_IA/`: `spot_republic_final_16x9.mp4`, `spot_republic_final_9x16.mp4` y `notas_del_editor.md`.

In [ ]:
#@title 1 · Instalar herramientas y montar Google Drive
#@markdown Usa **Entorno de ejecución → Cambiar tipo → GPU (T4 o mejor)**. Tarda ~2 min.
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'faster-whisper', 'librosa', 'ultralytics', 'open_clip_torch', 'anthropic>=0.70'], check=True)
    # vLLM va en un entorno aparte para no romper el torch de Colab (usado por CLIP/YOLO/Whisper)
    if not os.path.exists('/content/vllm_env/bin/vllm'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/vllm_env'], check=True)
        subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/vllm_env/bin/python', '-U', 'vllm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'Sin GPU (irá más lento)')
except FileNotFoundError:
    print('Sin GPU (irá más lento)')

In [ ]:
#@title 2 · Configuración del spot
import os, glob, json, math, random, re, shutil, subprocess, base64
import numpy as np, cv2

DRIVE_FOLDER   = "/content/drive/MyDrive/Server"  #@param {type:"string"}
SOUNDTRACK     = "spot_republic_soundtrack_master.mp4"  #@param {type:"string"}
SPOT_TITLE     = "SPOT REPUBLIC"  #@param {type:"string"}
DURACION_MAX_S = 45  #@param {type:"slider", min:15, max:120, step:5}
GRADE          = "teal_orange"  #@param ["teal_orange", "warm_film", "cool_night", "bw_noir"]
LETTERBOX      = True  #@param {type:"boolean"}
IDIOMA_SUBS    = "es"  #@param ["es", "en", "auto"]
SUBS_DESDE     = "auto"  #@param ["auto", "soundtrack", "clips", "ninguno"]
MEZCLA_AUDIO_CLIPS = 0.0  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown **Motor de IA para los "pensamientos del editor"** · `local_qwen` = Qwen3.8-27B (visión) corriendo en tu GPU con vLLM, sin API key (necesita ≥ 70 GB de VRAM: H100/A100 80 GB) · `claude` = API de Anthropic (secreto `ANTHROPIC_API_KEY`) · `frases` = frases fijas.
MOTOR_IA       = "local_qwen"  #@param ["local_qwen", "claude", "frases"]
MODELO_LOCAL   = "Qwen/Qwen3.8-27B"  #@param {type:"string"}
FPS = 30
SEED = 7
random.seed(SEED); np.random.seed(SEED)

def find_folder(start="/content/drive/MyDrive", max_depth=3):
    if os.path.exists(os.path.join(DRIVE_FOLDER, SOUNDTRACK)):
        return DRIVE_FOLDER
    base = start.rstrip('/').count('/')
    for root, dirs, files in os.walk(start):
        if SOUNDTRACK in files:
            return root
        if root.count('/') - base >= max_depth:
            dirs[:] = []
    raise FileNotFoundError(f"No encuentro {SOUNDTRACK} en Drive. Ajusta DRIVE_FOLDER.")

FOLDER = find_folder()
WORK = os.environ.get("SPOT_WORK", "/content/work")
OUT_DIR = os.path.join(FOLDER, "Edicion_IA")
for d in (WORK, f"{WORK}/proxy", f"{WORK}/frames", OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Carpeta:", FOLDER, "\nSalida:", OUT_DIR)

In [ ]:
#@title 3 · Inventario, quitar duplicados y crear proxies 1080p (tonemap HDR si hace falta)
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))[:300]}\n{r.stderr[-1500:]}")
    return r

def probe(path):
    r = run(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', path])
    j = json.loads(r.stdout)
    v = next((s for s in j['streams'] if s['codec_type'] == 'video'), None)
    a = next((s for s in j['streams'] if s['codec_type'] == 'audio'), None)
    return {'duration': float(j['format'].get('duration', 0)), 'video': v, 'audio': a}

def pick_encoder():
    try:
        run(['ffmpeg', '-v', 'error', '-f', 'lavfi', '-i', 'color=s=256x256:d=0.2', '-c:v', 'h264_nvenc', '-f', 'null', '-'])
        return ['-c:v', 'h264_nvenc', '-preset', 'p6', '-cq', '19', '-b:v', '0']
    except Exception:
        return ['-c:v', 'libx264', '-preset', 'medium', '-crf', '18']
VENC = pick_encoder()
print("Encoder:", VENC[1])

all_mp4 = sorted(glob.glob(os.path.join(FOLDER, '*.mp4')), key=lambda p: (re.sub(r' \(\d+\)(?=\.mp4$)', '', p), bool(re.search(r' \(\d+\)\.mp4$', p)), p))  # originales antes que 'nombre (1).mp4'
clips, seen = [], set()
for p in all_mp4:
    name = os.path.basename(p)
    if name == SOUNDTRACK or name.startswith('spot_republic') or os.path.getsize(p) == 0:
        continue
    info = probe(p)
    if info['video'] is None or info['duration'] < 1.5:
        continue
    key = (os.path.getsize(p), round(info['duration'], 2))  # copias subidas dos veces
    if key in seen:
        print("  duplicado, omitido:", name); continue
    seen.add(key)
    clips.append({'src': p, 'name': name, **info})
print(f"{len(clips)} clips únicos, {sum(c['duration'] for c in clips)/60:.1f} min de material")

HDR_TRC = {'arib-std-b67', 'smpte2084'}
def make_proxy(c, i):
    out = f"{WORK}/proxy/clip{i:02d}.mp4"
    c['proxy'] = out
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return
    v = c['video']
    scale = "scale='if(gt(iw,ih),-2,1080)':'if(gt(iw,ih),1080,-2)':flags=lanczos"
    chains = []
    if v.get('color_transfer') in HDR_TRC:   # DJI HLG/HDR -> SDR Rec.709
        chains.append("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
                      "tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p," + scale)
    chains.append(scale + ",format=yuv420p")
    for vf in chains:
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-i', c['src'], '-vf', f"{vf},fps={FPS}",
                 *VENC, '-g', str(FPS), '-c:a', 'aac', '-b:a', '192k', '-ac', '2', '-ar', '48000', out])
            return
        except RuntimeError as e:
            print("  reintentando sin tonemap:", str(e)[-200:])
    raise RuntimeError("No se pudo crear proxy de " + c['name'])

for i, c in enumerate(clips):
    print(f"proxy {i+1}/{len(clips)}: {c['name']}  ({c['duration']:.0f}s, {c['video']['width']}x{c['video']['height']}, trc={c['video'].get('color_transfer')})")
    make_proxy(c, i)
    cap = cv2.VideoCapture(c['proxy'])
    c['W'], c['H'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    c['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    c['has_audio'] = c['audio'] is not None

# Soundtrack -> wav
ST_WAV = f"{WORK}/soundtrack.wav"
run(['ffmpeg', '-y', '-v', 'error', '-i', os.path.join(FOLDER, SOUNDTRACK), '-vn', '-ac', '2', '-ar', '48000', ST_WAV])

In [ ]:
#@title 4 · Analizar el ritmo del soundtrack y crear la línea de cortes
import librosa
y, sr = librosa.load(ST_WAV, sr=22050, mono=True)
st_dur = len(y) / sr
SPOT_LEN = min(float(DURACION_MAX_S), st_dur)
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, units='frames')
tempo = float(np.atleast_1d(tempo)[0])
beats = librosa.frames_to_time(beat_frames, sr=sr)
onset_env = librosa.onset.onset_strength(y=y, sr=sr)
strength = onset_env[np.clip(beat_frames, 0, len(onset_env) - 1)] if len(beat_frames) else np.array([])

if len(beats) < 4 or tempo <= 0:          # sin ritmo claro: cortes cada 2.5 s
    beats = np.arange(0, SPOT_LEN, 1.25); strength = np.ones_like(beats); tempo = 96.0
BEATS_PER_CUT = 4 if tempo >= 110 else 2
print(f"Tempo ≈ {tempo:.0f} BPM · spot de {SPOT_LEN:.1f}s · corte cada {BEATS_PER_CUT} beats")

# Cortes: intro más larga (título), cortes en beats fuertes, cierre más largo
cuts = [0.0]
i = int(np.searchsorted(beats, 2.8))
while i < len(beats) and beats[i] < SPOT_LEN - 2.0:
    window = range(i, min(i + 2, len(beats)))           # elige el beat más fuerte cercano
    j = max(window, key=lambda k: strength[k] if k < len(strength) else 0)
    if beats[j] - cuts[-1] >= 1.0:
        cuts.append(float(beats[j]))
    i = j + BEATS_PER_CUT
cuts.append(SPOT_LEN)
frame_bounds = [round(t * FPS) for t in cuts]
SHOTS = [{'idx': k, 'f0': frame_bounds[k], 'f1': frame_bounds[k + 1],
          'dur': (frame_bounds[k + 1] - frame_bounds[k]) / FPS} for k in range(len(cuts) - 1)]
SHOTS = [s for s in SHOTS if s['f1'] > s['f0']]
TOTAL_FRAMES = SHOTS[-1]['f1']
print(f"{len(SHOTS)} planos:", [round(s['dur'], 2) for s in SHOTS])

In [ ]:
#@title 5 · IA: puntuar cada momento (CLIP estética + nitidez + exposición + estabilidad) y elegir los mejores planos
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    import open_clip
    BIG_GPU = DEVICE == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 30e9
    CLIP_ARCH, CLIP_W = ('ViT-H-14', 'laion2b_s32b_b79k') if BIG_GPU else ('ViT-B-32', 'laion2b_s34b_b79k')
    clip_model, _, clip_pre = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_W, device=DEVICE)
    clip_model.eval(); tok = open_clip.get_tokenizer(CLIP_ARCH); print("CLIP:", CLIP_ARCH)
    POS = ["a cinematic, beautifully composed film still", "a stunning shot with dramatic light",
           "people having fun at an event, candid moment", "a vibrant city or venue at night with lights"]
    NEG = ["a blurry, out of focus, shaky photo", "a dark underexposed photo", "a photo of the floor or a wall",
           "a camera pointed at nothing, accidental shot"]
    with torch.no_grad():
        T = clip_model.encode_text(tok(POS + NEG).to(DEVICE)); T = T / T.norm(dim=-1, keepdim=True)
    HAVE_CLIP = True
except Exception as e:
    print("CLIP no disponible, uso solo métricas clásicas:", e); HAVE_CLIP = False

from PIL import Image
STEP_S = 0.5
def analyze(c):
    cap = cv2.VideoCapture(c['proxy']); step = int(STEP_S * FPS)
    times, feats, prev = [], [], None
    imgs = []
    f = 0
    while True:
        ok = cap.grab()
        if not ok: break
        if f % step == 0:
            ok, fr = cap.retrieve()
            if not ok: break
            small = cv2.resize(fr, (320, int(320 * fr.shape[0] / fr.shape[1])))
            g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            sharp = cv2.Laplacian(g, cv2.CV_64F).var()
            bright = g.mean() / 255
            motion = 0.0 if prev is None else float(np.abs(g.astype(np.int16) - prev).mean() / 255)
            prev = g.astype(np.int16)
            times.append(f / FPS); feats.append((sharp, bright, motion))
            if HAVE_CLIP: imgs.append(clip_pre(Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))))
        f += 1
    cap.release()
    feats = np.array(feats) if feats else np.zeros((0, 3))
    sharp = np.log1p(feats[:, 0]); sharp = (sharp - sharp.min()) / (np.ptp(sharp) + 1e-6) if len(sharp) else sharp
    expo = 1 - np.clip(np.abs(feats[:, 1] - 0.45) / 0.45, 0, 1)
    shake = np.clip(feats[:, 2] / 0.12, 0, 1)                        # mucho movimiento = vibración
    score = 0.35 * sharp + 0.25 * expo + 0.15 * (1 - shake) + 0.25 * np.clip(feats[:, 2] / 0.03, 0, 1) * (1 - shake)
    if HAVE_CLIP and imgs:
        aest = []
        with torch.no_grad():
            for b in range(0, len(imgs), 64):
                I = clip_model.encode_image(torch.stack(imgs[b:b + 64]).to(DEVICE)); I = I / I.norm(dim=-1, keepdim=True)
                p = (100 * I @ T.T).softmax(dim=-1).cpu().numpy()
                aest.append(p[:, :len(POS)].sum(1))
        aest = np.concatenate(aest)
        score = 0.55 * aest + 0.45 * score
    c['t'] = np.array(times); c['score'] = np.convolve(score, np.ones(3) / 3, mode='same') if len(score) > 3 else score

for c in clips:
    analyze(c); print(f"  {c['name']}: mejor momento {c['t'][np.argmax(c['score'])]:.1f}s (score {c['score'].max():.2f})")

# Candidatos: picos con supresión de no-máximos (≥3 s entre sí)
cands = []
for ci, c in enumerate(clips):
    s = c['score'].copy(); t = c['t']
    for _ in range(40):
        if not len(s) or s.max() <= -1: break
        k = int(np.argmax(s)); cands.append((float(s[k]), ci, float(t[k])))
        s[np.abs(t - t[k]) < 3.0] = -2
cands.sort(reverse=True)
cap_per_clip = max(2, math.ceil(len(SHOTS) / max(1, len(clips))) + 1)
used, chosen = {}, []
for sc, ci, tc in cands:
    if len(chosen) >= len(SHOTS): break
    if used.get(ci, 0) >= cap_per_clip: continue
    used[ci] = used.get(ci, 0) + 1; chosen.append((ci, tc, sc))
while len(chosen) < len(SHOTS):                            # poco material: reutiliza
    chosen.append(chosen[len(chosen) % max(1, len(chosen))])
# Orden narrativo: cronológico (los nombres DJI llevan fecha/hora)
best = max(range(len(chosen)), key=lambda k: chosen[k][2])
hero = chosen.pop(best)                                    # el mejor plano abre con el título
chosen.sort(key=lambda x: (clips[x[0]]['name'], x[1]))
chosen.insert(0, hero)
for s, (ci, tc, sc) in zip(SHOTS, chosen):
    c = clips[ci]; n = s['f1'] - s['f0']
    start = int(round(tc * FPS - n / 2)); start = max(0, min(start, c['nframes'] - n - 1))
    s.update(clip=ci, src_f0=max(0, start), score=sc)
print("Plan de montaje listo.")
try:                                                       # libera VRAM para el modelo local
    del clip_model; torch.cuda.empty_cache()
except Exception:
    pass

In [ ]:
#@title 6 · IA: "pensamientos del editor" (modelo local Qwen3.8-27B en tu GPU, o Claude) mira cada plano y escribe textos y notas
def read_frame(ci, f):
    cap = cv2.VideoCapture(clips[ci]['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, fr = cap.read(); cap.release()
    return fr if ok else np.zeros((1080, 1920, 3), np.uint8)

thumbs = []
for s in SHOTS:
    fr = read_frame(s['clip'], s['src_f0'] + (s['f1'] - s['f0']) // 2)
    h, w = fr.shape[:2]; k = 768 / max(h, w)
    fr = cv2.resize(fr, (int(w * k), int(h * k)))
    p = f"{WORK}/frames/shot{s['idx']:02d}.jpg"; cv2.imwrite(p, fr, [cv2.IMWRITE_JPEG_QUALITY, 85]); thumbs.append(p)

FALLBACK = ["La noche apenas empieza.", "Aquí nadie mira el reloj.", "Esto no se planea, se vive.",
            "Luces, ruido, nosotros.", "Un segundo que dura para siempre.", "El ritmo nos encontró.",
            "Recuérdalo así.", "Todo pasa por algo."]
EDIT = {'titulo': SPOT_TITLE, 'subtitulo': '', 'cierre': 'Nos vemos en la próxima.',
        'planos': [{'indice': s['idx'], 'pensamiento': FALLBACK[k % len(FALLBACK)], 'mostrar': k % 3 == 1,
                    'nota_editor': ''} for k, s in enumerate(SHOTS)]}

PROMPT = (
    f"Eres el editor de un spot cinematográfico llamado '{SPOT_TITLE}' (idioma: español). Arriba están los planos "
    f"en orden de montaje, cortados al ritmo de la música, {SPOT_LEN:.0f}s en total. Escribe los textos en pantalla: "
    "'pensamiento' = un pensamiento íntimo del editor sobre ese plano, en primera persona o como voz interior, "
    "máximo 7 palabras, evocador, nada genérico, que conecte con lo que realmente se ve. Marca 'mostrar'=true en "
    "aproximadamente un tercio de los planos (los que más lo merezcan, nunca dos seguidos, nunca el plano 0, que lleva el título). "
    "'nota_editor' = una frase explicando por qué ese plano está ahí y qué emoción aporta. Propón también un subtítulo "
    "corto para el título y una frase de cierre (máx. 6 palabras). Responde solo con el JSON pedido.")
SCHEMA = {'type': 'object', 'additionalProperties': False,
          'required': ['titulo', 'subtitulo', 'cierre', 'planos'],
          'properties': {'titulo': {'type': 'string'}, 'subtitulo': {'type': 'string'}, 'cierre': {'type': 'string'},
                         'planos': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                    'required': ['indice', 'pensamiento', 'mostrar', 'nota_editor'],
                                    'properties': {'indice': {'type': 'integer'}, 'pensamiento': {'type': 'string'},
                                                   'mostrar': {'type': 'boolean'}, 'nota_editor': {'type': 'string'}}}}}}
B64 = [base64.standard_b64encode(open(p, 'rb').read()).decode() for p in thumbs]

def apply_result(data, motor):
    global EDIT
    data['titulo'] = SPOT_TITLE or data['titulo']
    byidx = {p['indice']: p for p in data['planos']}
    data['planos'] = [byidx.get(s['idx'], EDIT['planos'][k]) for k, s in enumerate(SHOTS)]
    EDIT = data
    print(f"{motor} escribió los pensamientos del editor ✓")

def run_local_qwen():
    """Sirve el modelo de visión en la GPU con vLLM (API compatible OpenAI) y le pasa los fotogramas."""
    import urllib.request, time as _t
    from openai import OpenAI
    vllm_bin = '/content/vllm_env/bin/vllm'
    if not os.path.exists(vllm_bin):
        raise RuntimeError("vLLM no está instalado (corre la celda 1 en Colab).")
    log = open(f"{WORK}/vllm.log", 'w')
    proc = subprocess.Popen([vllm_bin, 'serve', MODELO_LOCAL, '--port', '8000', '--max-model-len', '32768',
                             '--gpu-memory-utilization', '0.88', '--limit-mm-per-prompt', json.dumps({'image': 64}),
                             '--served-model-name', 'local'], stdout=log, stderr=subprocess.STDOUT)
    print(f"Descargando y cargando {MODELO_LOCAL} en la GPU (la 1.ª vez ~5-15 min)…")
    t0 = _t.time()
    while True:
        try:
            urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); break
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError("vLLM se detuvo:\n" + open(f"{WORK}/vllm.log").read()[-3000:])
            if _t.time() - t0 > 40 * 60:
                proc.terminate(); raise RuntimeError("vLLM tardó demasiado en arrancar")
            _t.sleep(10)
    print(f"Modelo listo en {(_t.time() - t0) / 60:.1f} min")
    try:
        client = OpenAI(base_url='http://localhost:8000/v1', api_key='local')
        content = []
        for s, b in zip(SHOTS, B64):
            content.append({'type': 'text', 'text': f"Plano {s['idx']} · {s['dur']:.1f}s"})
            content.append({'type': 'image_url', 'image_url': {'url': f"data:image/jpeg;base64,{b}"}})
        content.append({'type': 'text', 'text': PROMPT})
        r = client.chat.completions.create(
            model='local', messages=[{'role': 'user', 'content': content}], max_tokens=4000,
            temperature=0.7, top_p=0.8, presence_penalty=1.5,
            response_format={'type': 'json_schema', 'json_schema': {'name': 'edicion', 'schema': SCHEMA}},
            extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}})
        txt = r.choices[0].message.content.strip()
        txt = txt[txt.find('{'): txt.rfind('}') + 1]
        apply_result(json.loads(txt), MODELO_LOCAL)
    finally:
        proc.terminate()                       # libera la VRAM para Whisper y el render
        try: proc.wait(timeout=60)
        except Exception: proc.kill()

def run_claude():
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        key = os.environ.get('ANTHROPIC_API_KEY')
    if not key:
        raise RuntimeError("Falta el secreto ANTHROPIC_API_KEY")
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    content = []
    for s, b in zip(SHOTS, B64):
        content.append({'type': 'text', 'text': f"Plano {s['idx']} · {s['dur']:.1f}s"})
        content.append({'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b}})
    content.append({'type': 'text', 'text': PROMPT})
    with client.beta.messages.stream(
        model='claude-opus-5-5', max_tokens=16000,
        betas=['server-side-fallback-2026-07-01'], fallbacks='default',
        output_config={'effort': 'high', 'format': {'type': 'json_schema', 'schema': SCHEMA}},
        messages=[{'role': 'user', 'content': content}],
    ) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == 'refusal':
        raise RuntimeError("Claude declinó la petición")
    apply_result(json.loads(next(b.text for b in msg.content if b.type == 'text')), 'Claude')

try:
    if MOTOR_IA == 'local_qwen': run_local_qwen()
    elif MOTOR_IA == 'claude': run_claude()
    else: print("Usando frases fijas (puedes editarlas en EDIT['planos']).")
except Exception as e:
    print(f"⚠ {MOTOR_IA} falló, uso frases de respaldo:", str(e)[-1500:])

EDIT['planos'][0]['mostrar'] = False
for p in EDIT['planos']:
    print(f"  {p['indice']:>2} {'👁' if p['mostrar'] else '  '} {p['pensamiento']}")

In [ ]:
#@title 7 · Audio final + subtítulos con IA (Whisper, palabra por palabra)
# Pista con el audio original de los planos elegidos (por si hay voces)
CLIPS_WAV = f"{WORK}/clips_audio.wav"
parts = []
for s in SHOTS:
    c = clips[s['clip']]; p = f"{WORK}/frames/a{s['idx']:02d}.wav"
    src = ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo'] if not c['has_audio'] else ['-ss', f"{s['src_f0']/FPS:.3f}", '-i', c['proxy']]
    run(['ffmpeg', '-y', '-v', 'error', *src, '-t', f"{(s['f1']-s['f0'])/FPS:.3f}", '-vn', '-ac', '2', '-ar', '48000',
         '-af', 'apad', '-t', f"{(s['f1']-s['f0'])/FPS:.3f}", p])
    parts.append(p)
with open(f"{WORK}/alist.txt", 'w') as fh:
    fh.writelines(f"file '{p}'\n" for p in parts)
run(['ffmpeg', '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', f"{WORK}/alist.txt", '-c', 'copy', CLIPS_WAV])

L = TOTAL_FRAMES / FPS
MIX_WAV = f"{WORK}/mix.wav"
fade = f"afade=t=in:d=0.3,afade=t=out:st={max(0, L-1.8):.3f}:d=1.8"
run(['ffmpeg', '-y', '-v', 'error', '-i', ST_WAV, '-i', CLIPS_WAV, '-filter_complex',
     f"[0:a]atrim=0:{L:.3f},asetpts=N/SR/TB[m];[1:a]volume={MEZCLA_AUDIO_CLIPS}[c];"
     f"[m][c]amix=inputs=2:normalize=0:duration=first,{fade},alimiter=limit=0.95[o]",
     '-map', '[o]', '-ar', '48000', MIX_WAV])

WORDS = []
if SUBS_DESDE != 'ninguno':
    from faster_whisper import WhisperModel
    try:
        wm = WhisperModel('large-v3', device='cuda', compute_type='float16')
    except Exception:
        wm = WhisperModel('small', device='cpu', compute_type='int8')
    def transcribe(path):
        segs, _ = wm.transcribe(path, language=None if IDIOMA_SUBS == 'auto' else IDIOMA_SUBS,
                                word_timestamps=True, vad_filter=True)
        return [(w.start, w.end, w.word.strip()) for sg in segs for w in (sg.words or []) if w.word.strip() and w.start < L]
    order = {'soundtrack': [MIX_WAV], 'clips': [CLIPS_WAV], 'auto': [MIX_WAV, CLIPS_WAV]}[SUBS_DESDE]
    for src in order:
        WORDS = transcribe(src)
        print(f"Whisper en {os.path.basename(src)}: {len(WORDS)} palabras")
        if len(WORDS) >= 6: break
    if len(WORDS) < 6:
        print("No hay voz/letra suficiente: el spot va sin subtítulos (sí con textos del editor)."); WORDS = []

def ass_time(t):
    t = max(0, t); h = int(t // 3600); m = int(t % 3600 // 60); s = t % 60
    return f"{h}:{m:02d}:{s:05.2f}"

def write_ass(path, W, H, margin_v, size):
    chunks, cur = [], []
    for w in WORDS:
        if cur and (len(cur) >= 4 or w[0] - cur[-1][1] > 0.6 or w[1] - cur[0][0] > 2.2):
            chunks.append(cur); cur = []
        cur.append(w)
    if cur: chunks.append(cur)
    lines = ["[Script Info]", "ScriptType: v4.00+", f"PlayResX: {W}", f"PlayResY: {H}", "WrapStyle: 2", "",
             "[V4+ Styles]",
             "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding",
             f"Style: K,Bebas Neue,{size},&H0030D5FF,&H00FFFFFF,&H00101010,&H64000000,0,0,0,0,100,100,2,0,1,3,2,2,60,60,{margin_v},1",
             "", "[Events]", "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text"]
    for ch in chunks:
        st, en = ch[0][0], max(ch[-1][1], ch[0][0] + 0.5)
        kar = "".join(f"{{\\k{max(1, int(round((b - a) * 100)))}}}{txt.upper()} " for a, b, txt in ch)
        lines.append(f"Dialogue: 0,{ass_time(st)},{ass_time(en)},K,,0,0,0,,{{\\fad(80,80)}}{kar.strip()}")
    open(path, 'w', encoding='utf-8').write("\n".join(lines) + "\n")

In [ ]:
#@title 8 · Look cinematográfico: color, grano, viñeta, textos en perspectiva 3D y reencuadre IA
from PIL import ImageDraw, ImageFont, ImageFilter
import urllib.request
FONT_DIR = f"{WORK}/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
FONT_URLS = {'BebasNeue-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/bebasneue/BebasNeue-Regular.ttf',
             'Lora-Italic.ttf': 'https://github.com/google/fonts/raw/main/ofl/lora/Lora-Italic%5Bwght%5D.ttf'}
for fn, url in FONT_URLS.items():
    p = os.path.join(FONT_DIR, fn)
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(url, p)
        except Exception as e: print("No pude bajar", fn, e)
def font(fn, size):
    for cand in (os.path.join(FONT_DIR, fn), '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'):
        try: return ImageFont.truetype(cand, size)
        except Exception: pass
    return ImageFont.load_default(size=size)

# ---------- Color grading (LUT por canal + saturación + split-toning) ----------
def curve(points):
    xs, ys = zip(*points); return np.clip(np.interp(np.arange(256), xs, ys), 0, 255).astype(np.uint8)
GRADES = {   # (curva B, curva G, curva R, saturación)
    'teal_orange': (curve([(0, 22), (64, 74), (128, 126), (192, 170), (255, 236)]),
                    curve([(0, 10), (64, 60), (128, 128), (192, 196), (255, 248)]),
                    curve([(0, 6), (64, 52), (128, 134), (192, 208), (255, 255)]), 1.12),
    'warm_film':   (curve([(0, 18), (128, 116), (255, 228)]), curve([(0, 14), (128, 128), (255, 244)]),
                    curve([(0, 16), (128, 140), (255, 252)]), 0.95),
    'cool_night':  (curve([(0, 28), (128, 142), (255, 255)]), curve([(0, 12), (128, 126), (255, 240)]),
                    curve([(0, 4), (128, 112), (255, 230)]), 0.9),
    'bw_noir':     (curve([(0, 0), (60, 40), (190, 210), (255, 255)]),) * 3 + (0.0,),
}
LB, LG, LR, SAT = GRADES[GRADE]
_vig_cache, _grain = {}, [np.random.normal(0, 5.5, (540, 540)).astype(np.int16) for _ in range(6)]
def grade(fr, fidx):
    h, w = fr.shape[:2]
    if SAT == 0.0:
        g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY); fr = cv2.cvtColor(cv2.LUT(g, LB), cv2.COLOR_GRAY2BGR)
    else:
        b, g, r = cv2.split(fr); fr = cv2.merge([cv2.LUT(b, LB), cv2.LUT(g, LG), cv2.LUT(r, LR)])
        if SAT != 1.0:
            hsv = cv2.cvtColor(fr, cv2.COLOR_BGR2HSV); hsv[..., 1] = cv2.convertScaleAbs(hsv[..., 1], alpha=SAT)
            fr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    if (w, h) not in _vig_cache:
        yy, xx = np.mgrid[0:h, 0:w]; d = np.sqrt(((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2)
        _vig_cache[(w, h)] = (1 - 0.32 * np.clip(d - 0.45, 0, 1) ** 1.6)[..., None].astype(np.float32)
    fr = (fr.astype(np.float32) * _vig_cache[(w, h)])
    n = cv2.resize(_grain[fidx % len(_grain)], (w, h), interpolation=cv2.INTER_LINEAR)
    return np.clip(fr + n[..., None], 0, 255).astype(np.uint8)

# ---------- Texto en perspectiva (rotación 3D en Y/X + proyección) ----------
def text_rgba(lines, fn, size, color=(255, 255, 255), tracking=0, shadow=True):
    f = font(fn, size); pad = int(size * 0.6)
    boxes = [f.getbbox(t) for t in lines]
    W = max(b[2] - b[0] + tracking * len(t) for b, t in zip(boxes, lines)) + 2 * pad
    lh = int(size * 1.15); H = lh * len(lines) + 2 * pad
    img = Image.new('RGBA', (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
    for k, t in enumerate(lines):
        x = pad
        for ch in t:
            d.text((x, pad + k * lh), ch, font=f, fill=color + (255,)); x += f.getlength(ch) + tracking
    if shadow:
        a = img.split()[-1].filter(ImageFilter.GaussianBlur(size / 10))
        sh = Image.new('RGBA', img.size, (0, 0, 0, 0)); sh.putalpha(a.point(lambda v: int(v * 0.75)))
        img = Image.alpha_composite(sh, img)
    return np.array(img)  # RGBA

def perspective_paste(frame, rgba, cx, cy, yaw=0.0, pitch=0.0, scale=1.0, alpha=1.0, focal=1400):
    h, w = rgba.shape[:2]
    pts = np.array([[-w / 2, -h / 2, 0], [w / 2, -h / 2, 0], [w / 2, h / 2, 0], [-w / 2, h / 2, 0]], np.float32) * scale
    cy_, sy = math.cos(math.radians(yaw)), math.sin(math.radians(yaw))
    cp, sp = math.cos(math.radians(pitch)), math.sin(math.radians(pitch))
    R = np.array([[cy_, 0, sy], [0, 1, 0], [-sy, 0, cy_]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
    P = pts @ R.T; z = P[:, 2] + focal
    dst = np.stack([cx + P[:, 0] * focal / z, cy + P[:, 1] * focal / z], 1).astype(np.float32)
    src = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(rgba, M, (frame.shape[1], frame.shape[0]), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
    a = (warped[..., 3:4].astype(np.float32) / 255) * alpha
    frame[:] = (frame * (1 - a) + warped[..., [2, 1, 0]] * a).astype(np.uint8)

def ease(x): x = min(max(x, 0), 1); return 1 - (1 - x) ** 3
_tcache = {}
def overlays(frame, s, local_f, fmt):
    W, H = frame.shape[1], frame.shape[0]; n = s['f1'] - s['f0']; t = local_f / FPS; T = n / FPS
    fade = min(ease(t / 0.45), ease((T - t) / 0.35))
    k = 1.0 if fmt == '16x9' else 0.78
    plan = EDIT['planos'][s['idx']]
    if s['idx'] == 0:                                     # Título que gira hacia cámara
        key = ('title', fmt)
        if key not in _tcache:
            _tcache[key] = (text_rgba([EDIT['titulo'].upper()], 'BebasNeue-Regular.ttf', int(190 * k), tracking=int(14 * k)),
                            text_rgba([EDIT.get('subtitulo', '')], 'Lora-Italic.ttf', int(46 * k)) if EDIT.get('subtitulo') else None)
        big, sub = _tcache[key]; e = ease(t / 1.4); fit = min(1.0, 0.86 * W / big.shape[1])
        perspective_paste(frame, big, W / 2, H * 0.47, yaw=-38 * (1 - e) + 6 * math.sin(t * 0.8),
                          pitch=10 * (1 - e), scale=fit * (0.85 + 0.15 * e + 0.02 * t), alpha=fade)
        if sub is not None:
            perspective_paste(frame, sub, W / 2, H * 0.47 + 120 * k, yaw=-8 * (1 - e), alpha=fade * ease((t - 0.6) / 0.8))
    elif s['idx'] == len(SHOTS) - 1:                      # Cierre
        key = ('end', fmt)
        if key not in _tcache:
            _tcache[key] = text_rgba([EDIT['cierre']], 'Lora-Italic.ttf', int(78 * k))
        fit = min(1.0, 0.86 * W / _tcache[key].shape[1])
        perspective_paste(frame, _tcache[key], W / 2, H * 0.5, yaw=14 * (1 - ease(t / 2)), pitch=-6,
                          scale=fit * (1 + 0.03 * t), alpha=fade)
    elif plan.get('mostrar'):                            # Pensamiento del editor, inclinado en el espacio
        key = ('th', s['idx'], fmt)
        side = -1 if s['idx'] % 2 else 1
        if key not in _tcache:
            words = plan['pensamiento'].split(); mid = (len(words) + 1) // 2
            lines = [' '.join(words[:mid]), ' '.join(words[mid:])] if len(words) > 3 else [plan['pensamiento']]
            kt = 1.0 if fmt == '16x9' else 1.15
            _tcache[key] = (text_rgba(['— nota del editor'], 'BebasNeue-Regular.ttf', int(34 * kt), color=(255, 205, 140), tracking=4),
                            text_rgba(lines, 'Lora-Italic.ttf', int(64 * kt)))
        lab, txt = _tcache[key]
        if txt.shape[1] > 0.9 * W:
            r = 0.9 * W / txt.shape[1]; txt = cv2.resize(txt, None, fx=r, fy=r, interpolation=cv2.INTER_AREA)
        cx = W * (0.5 - side * 0.2) if fmt == '16x9' else W / 2
        cy = H * 0.62 if fmt == '16x9' else H * 0.30
        yaw = side * (28 - 10 * ease(t / T)); drift = 20 * t
        perspective_paste(frame, lab, cx - side * drift * 0.3, cy - txt.shape[0] * 0.55, yaw=yaw, pitch=4, alpha=fade)
        perspective_paste(frame, txt, cx - side * drift * 0.3, cy, yaw=yaw, pitch=4, alpha=fade)

# ---------- Reencuadre inteligente 9:16 (YOLO sigue a las personas) ----------
try:
    from ultralytics import YOLO
    yolo = YOLO('yolo11n.pt')
except Exception as e:
    print("YOLO no disponible, reencuadre centrado:", e); yolo = None

def track_centers(s):
    """Centro X normalizado (0-1) por frame del plano, suavizado."""
    c = clips[s['clip']]; n = s['f1'] - s['f0']
    if yolo is None:
        return np.full(n, 0.5)
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, s['src_f0'])
    xs, ts = [], []
    for i in range(n):
        ok, fr = cap.read()
        if not ok: break
        if i % 5 == 0:
            r = yolo.predict(fr, classes=[0], conf=0.35, verbose=False, imgsz=640)[0]
            if len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy(); area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
                wts = area / area.sum(); x = float(((b[:, 0] + b[:, 2]) / 2 * wts).sum() / fr.shape[1])
                xs.append(x); ts.append(i)
    cap.release()
    if not xs:
        return np.full(n, 0.5)
    raw = np.interp(np.arange(n), ts, xs)
    out = np.empty(n); out[0] = raw[0]
    for i in range(1, n): out[i] = out[i - 1] * 0.92 + raw[i] * 0.08   # cámara suave
    return out

for s in SHOTS:
    c = clips[s['clip']]
    s['cx'] = track_centers(s) if c['W'] > c['H'] else np.full(s['f1'] - s['f0'], 0.5)
print("Look y reencuadre preparados.")

In [ ]:
#@title 9 · Render final: 16:9 cinematográfico y 9:16 vertical
import time
FORMATS = {'16x9': (1920, 1080), '9x16': (1080, 1920)}
BAR = int(round((1080 - 1920 / 2.39) / 2))   # barras 2.39:1

def crop_frame(fr, W, H, cxn, zoom):
    h, w = fr.shape[:2]; tgt = W / H
    cw, ch = (h * tgt, h) if w / h > tgt else (w, w / tgt)
    cw, ch = cw / zoom, ch / zoom
    x0 = np.clip(cxn * w - cw / 2, 0, w - cw); y0 = (h - ch) / 2
    M = np.float32([[W / cw, 0, -x0 * W / cw], [0, H / ch, -y0 * H / ch]])
    return cv2.warpAffine(fr, M, (W, H), flags=cv2.INTER_AREA if cw > W else cv2.INTER_CUBIC)

def render(fmt):
    W, H = FORMATS[fmt]; ass = f"{WORK}/subs_{fmt}.ass"
    if WORDS:
        write_ass(ass, W, H, margin_v=(BAR + 40 if (fmt == '16x9' and LETTERBOX) else 70) if fmt == '16x9' else 560,
                  size=64 if fmt == '16x9' else 96)
    out = os.path.join(WORK, f"spot_republic_final_{fmt}.mp4")
    vf = [f"ass={os.path.basename(ass)}:fontsdir={FONT_DIR}"] if WORDS else []
    cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{W}x{H}', '-r', str(FPS), '-i', '-',
           '-i', MIX_WAV, *(['-vf', ','.join(vf)] if vf else []), '-map', '0:v', '-map', '1:a',
           *VENC, '-pix_fmt', 'yuv420p', '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
           '-c:a', 'aac', '-b:a', '256k', '-shortest', '-movflags', '+faststart', out]
    pipe = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE, cwd=WORK)
    t0 = time.time(); g = 0
    for s in SHOTS:
        c = clips[s['clip']]; n = s['f1'] - s['f0']
        cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, s['src_f0']); last = None
        z0, z1 = (1.0, 1.07) if s['idx'] % 2 == 0 else (1.07, 1.0)     # push-in / pull-out alterno
        for i in range(n):
            ok, fr = cap.read()
            fr = fr if ok else (last if last is not None else np.zeros((c['H'], c['W'], 3), np.uint8)); last = fr
            cxn = s['cx'][min(i, len(s['cx']) - 1)] if fmt == '9x16' else 0.5
            f = crop_frame(fr, W, H, cxn, z0 + (z1 - z0) * (i / max(1, n - 1)))
            f = grade(f, g)
            if i < 3 and s['idx'] > 0 and s['idx'] % 4 == 0:              # destello en golpes fuertes
                f = cv2.addWeighted(f, 1.0, np.full_like(f, 255), 0.35 * (1 - i / 3), 0)
            overlays(f, s, i, fmt)
            if fmt == '16x9' and LETTERBOX:
                f[:BAR] = 0; f[H - BAR:] = 0
            if s['idx'] == 0 and i < 12: f = (f * (i / 12)).astype(np.uint8)            # fade-in
            if s['idx'] == len(SHOTS) - 1 and n - i < 18: f = (f * ((n - i) / 18)).astype(np.uint8)  # fade-out
            pipe.stdin.write(f.tobytes()); g += 1
        cap.release()
        print(f"\r  {fmt}: {g}/{TOTAL_FRAMES} frames ({g / max(1e-6, time.time() - t0):.1f} fps)", end='')
    pipe.stdin.close(); err = pipe.stderr.read().decode(); pipe.wait()
    if pipe.returncode != 0:
        raise RuntimeError(err[-2000:])
    print(f"\n  ✓ {out}")
    return out

OUTS = [render(fmt) for fmt in FORMATS]

In [ ]:
#@title 10 · Guardar en Drive (carpeta Edicion_IA) + notas del editor
notes = [f"# {EDIT['titulo']} — notas del editor", "", f"*{EDIT.get('subtitulo', '')}*", "",
         f"- Duración: {TOTAL_FRAMES / FPS:.1f}s · {len(SHOTS)} planos · tempo ≈ {tempo:.0f} BPM · grade: {GRADE}",
         f"- Subtítulos: {'sí, ' + str(len(WORDS)) + ' palabras' if WORDS else 'no (sin voz detectada)'}", "",
         "| # | Tiempo | Clip | Pensamiento en pantalla | Por qué está aquí |", "|---|---|---|---|---|"]
for s, p in zip(SHOTS, EDIT['planos']):
    c = clips[s['clip']]
    notes.append(f"| {s['idx']} | {s['f0']/FPS:.1f}–{s['f1']/FPS:.1f}s | {c['name']} @ {s['src_f0']/FPS:.1f}s | "
                 f"{p['pensamiento'] if p.get('mostrar') else '—'} | {p.get('nota_editor', '')} |")
notes += ["", f"Cierre: *{EDIT['cierre']}*"]
open(f"{WORK}/notas_del_editor.md", 'w', encoding='utf-8').write("\n".join(notes))

for p in OUTS + [f"{WORK}/notas_del_editor.md"]:
    shutil.copy(p, OUT_DIR); print("→", os.path.join(OUT_DIR, os.path.basename(p)))

if IN_COLAB:
    from IPython.display import Video, display
    prev = f"{WORK}/preview.mp4"
    run(['ffmpeg', '-y', '-v', 'error', '-i', OUTS[0], '-vf', 'scale=640:-2', '-c:v', 'libx264', '-crf', '30', '-c:a', 'aac', '-b:a', '96k', prev])
    display(Video(prev, embed=True, width=720))
print("\n¿Cambiar algo? Edita EDIT['planos'][i]['pensamiento'] / ['mostrar'], o GRADE, y vuelve a correr las celdas 8–10.")